# MMMData Behavioral Analysis

This notebook provides a step-by-step walkthrough of the behavioral analyses for the **Multi-Modal Memory Data** (MMMData) project.

## Dataset Overview

- **Subjects**: sub-03, sub-04, sub-05
- **Sessions**: 29-30 sessions per subject
- **Trial-Based (TB) sessions**: ses-04 through ses-18 (15 recognition sessions)
- **Final session**: ses-30 (comprehensive recognition + temporal judgment)

### Task Structure

| Task | Description | Sessions |
|------|-------------|----------|
| **TBencoding** | Stimulus encoding with quality ratings (1-3) | ses-04 to ses-17 |
| **TBretrieval** | Cued retrieval with vividness ratings (1-3) | ses-04 to ses-18 |
| **TB2AFC** | 2-alternative forced choice recognition | ses-04 to ses-18 |
| **FIN2AFC** | Final comprehensive recognition test | ses-30 |
| **FINtimeline** | Temporal judgment (when was it seen?) | ses-30 |

### Experimental Conditions

- **enCon** (Encoding Condition): 1=single, 2=repeats, 3=triplets
- **reCon** (Retrieval Condition): 1=within-session, 2=across-session

### 2AFC Response Scheme

| Response | Position | Confidence |
|----------|----------|------------|
| 1 | Image 1 (left) | Sure |
| 2 | Image 1 (left) | Maybe |
| 3 | Image 2 (right) | Maybe |
| 4 | Image 2 (right) | Sure |

## Setup

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.3f}".format)

In [4]:
# Add project source to path and import behavioral analysis library
CODE_ROOT = Path("/gpfs/projects/hulacon/shared/mmmdata/code/mmmdata")
sys.path.insert(0, str(CODE_ROOT / "src" / "python"))

from behavioral import io, preprocessing, accuracy, rt, learning, encoding, final_session, plotting
from behavioral.constants import (
    SUBJECT_IDS, TB_SESSIONS, TB_ENCODING_SESSIONS,
    ENCON_LABELS, RECON_LABELS, SESSION_ORDER,
)

BIDS_ROOT = Path("/gpfs/projects/hulacon/shared/mmmdata")

print(f"BIDS root: {BIDS_ROOT}")
print(f"Subjects:  {SUBJECT_IDS}")
print(f"TB sessions: ses-{TB_SESSIONS[0]} to ses-{TB_SESSIONS[-1]} ({len(TB_SESSIONS)} sessions)")

BIDS root: /gpfs/projects/hulacon/shared/mmmdata
Subjects:  ('03', '04', '05')
TB sessions: ses-04 to ses-18 (15 sessions)


---
## 1. Data Inventory

First, let's verify which behavioral files are present on disk and get an overview of the dataset.

In [5]:
# Discover all behavioral files
tb_files = io.find_tb2afc_files(BIDS_ROOT)
enc_files = io.find_encoding_files(BIDS_ROOT)
ret_files = io.find_retrieval_files(BIDS_ROOT)
fin_files = io.find_fin2afc_files(BIDS_ROOT)
tl_files = io.find_fintimeline_files(BIDS_ROOT)

print("=== File Inventory ===")
print(f"TB2AFC files:      {len(tb_files):>3d}  (3 subjects x 15 sessions = 45 expected)")
print(f"TBencoding files:  {len(enc_files):>3d}  (multiple runs per session)")
print(f"TBretrieval files: {len(ret_files):>3d}  (multiple runs per session)")
print(f"FIN2AFC files:     {len(fin_files):>3d}  (1 per subject = 3 expected)")
print(f"FINtimeline files: {len(tl_files):>3d}  (1 per subject = 3 expected)")

=== File Inventory ===
TB2AFC files:       45  (3 subjects x 15 sessions = 45 expected)
TBencoding files:  126  (multiple runs per session)
TBretrieval files: 168  (multiple runs per session)
FIN2AFC files:       3  (1 per subject = 3 expected)
FINtimeline files:   3  (1 per subject = 3 expected)


In [6]:
# Check coverage per subject
for sub in SUBJECT_IDS:
    sub_tb = [f for f in tb_files if f"sub-{sub}" in str(f)]
    sub_enc = [f for f in enc_files if f"sub-{sub}" in str(f)]
    sub_ret = [f for f in ret_files if f"sub-{sub}" in str(f)]
    sub_fin = [f for f in fin_files if f"sub-{sub}" in str(f)]
    print(f"\nsub-{sub}: TB2AFC={len(sub_tb)}, encoding={len(sub_enc)}, "
          f"retrieval={len(sub_ret)}, FIN2AFC={len(sub_fin)}")


sub-03: TB2AFC=15, encoding=42, retrieval=56, FIN2AFC=1

sub-04: TB2AFC=15, encoding=42, retrieval=56, FIN2AFC=1

sub-05: TB2AFC=15, encoding=42, retrieval=56, FIN2AFC=1


---
## 2. Load Data

Load all behavioral data into DataFrames. The `io` module handles BIDS filename parsing, column normalization, and concatenation across files.

In [7]:
# Load TB2AFC recognition data (main behavioral dataset)
tb2afc = io.load_tb2afc(BIDS_ROOT)
print(f"TB2AFC: {len(tb2afc):,} trials")
print(f"  Subjects: {sorted(tb2afc['subject'].unique())}")
print(f"  Sessions: {sorted(tb2afc['session'].unique())}")
print(f"  Columns:  {list(tb2afc.columns)}")
tb2afc.head()

TB2AFC: 3,234 trials
  Subjects: ['03', '04', '05']
  Sessions: ['04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18']
  Columns:  ['onset', 'duration', 'run', 'trial_type', 'modality', 'word', 'image1', 'image2', 'correct_resp', 'resp', 'resp_RT', 'trial_accuracy', 'enCon', 'reCon', 'cueId', 'pairId', 'recog', 'trial_id', 'subject', 'session', 'source_file']


,onset,duration,run,trial_type,modality,word,image1,image2,correct_resp,resp,resp_RT,trial_accuracy,enCon,reCon,cueId,pairId,recog,trial_id,subject,session,source_file
0,0.000,2.364,01,recognition,visual,province,shared0176_nsd14611.png,shared0695_nsd50812.png,1,1,2.364,1.000,1,1,1.000,36,1.000,1,03,04,/gpfs/projects/hulacon/shared/mmmdata/sub-03/s...
1,2.364,3.221,01,recognition,visual,erect,shared0534_nsd40841.png,shared0244_nsd20224.png,2,4,3.221,1.000,1,1,1.000,32,2.000,2,03,04,/gpfs/projects/hulacon/shared/mmmdata/sub-03/s...
2,5.585,1.563,01,recognition,visual,father,shared0217_nsd17777.png,shared0291_nsd22994.png,1,1,1.563,1.000,3,1,1.000,11,1.000,3,03,04,/gpfs/projects/hulacon/shared/mmmdata/sub-03/s...
3,7.149,1.399,01,recognition,visual,shower,shared0418_nsd31029.png,shared0195_nsd16422.png,1,1,1.399,1.000,1,1,1.000,28,1.000,4,03,04,/gpfs/projects/hulacon/shared/mmmdata/sub-03/s...
4,8.547,2.037,01,recognition,visual,market,shared1000_nsd72949.png,shared0571_nsd42913.png,2,4,2.037,1.000,1,1,1.000,31,2.000,5,03,04,/gpfs/projects/hulacon/shared/mmmdata/sub-03/s...


In [8]:
# Load encoding event data
enc_df = io.load_encoding(BIDS_ROOT)
enc_df = preprocessing.remap_scanner_resp(enc_df)  # 6,7,8 -> 1,2,3
print(f"Encoding: {len(enc_df):,} trials")
print(f"  Response values after remapping: {sorted(enc_df['resp'].dropna().unique())}")
enc_df.head()

Encoding: 7,686 trials
  Response values after remapping: [1.0, 2.0, 3.0]


,onset,duration,onset_actual,duration_actual,subject,session,run,trial_type,modality,word,pairId,mmmId,nsdId,itmno,sharedId,voiceId,voice,enCon,reCon,resp,resp_RT,trial_id,source_file
0,9.000,3.000,9.020,3.008,03,04,01,image,visual,theater,1.000,998.000,72606.000,909.000,1.000,1.000,echo,3.000,1.000,3.000,1.648,1,/gpfs/projects/hulacon/shared/mmmdata/derivati...
1,13.500,3.000,13.520,3.000,03,04,01,image,visual,angel,2.000,1000.000,72949.000,38.000,1.000,3.000,onyx,3.000,1.000,2.000,1.484,2,/gpfs/projects/hulacon/shared/mmmdata/derivati...
2,18.000,3.000,18.020,3.001,03,04,01,image,visual,create,3.000,999.000,72720.000,236.000,1.000,3.000,onyx,3.000,1.000,2.000,1.792,3,/gpfs/projects/hulacon/shared/mmmdata/derivati...
3,22.500,3.000,22.523,3.003,03,04,01,image,visual,cunning,13.000,757.000,55108.000,242.000,0.000,2.000,nova,3.000,1.000,3.000,1.450,4,/gpfs/projects/hulacon/shared/mmmdata/derivati...
4,27.000,3.000,27.037,3.000,03,04,01,image,visual,convince,14.000,832.000,60506.000,220.000,0.000,3.000,onyx,3.000,1.000,1.000,2.176,5,/gpfs/projects/hulacon/shared/mmmdata/derivati...


In [9]:
# Load final session data
fin2afc = io.load_fin2afc(BIDS_ROOT)
timeline = io.load_fintimeline(BIDS_ROOT)
print(f"FIN2AFC:     {len(fin2afc):,} trials")
print(f"FINtimeline: {len(timeline):,} trials")

FIN2AFC:     720 trials
FINtimeline: 720 trials


### 2.1 Data Validation

Run validation checks to ensure expected columns and value ranges.

In [10]:
for name, df, task in [
    ("TB2AFC", tb2afc, "tb2afc"),
    ("Encoding", enc_df, "encoding"),
    ("FIN2AFC", fin2afc, "fin2afc"),
    ("FINtimeline", timeline, "fintimeline"),
]:
    warnings = preprocessing.validate_dataframe(df, task)
    status = "PASS" if not warnings else "WARNINGS"
    print(f"{name}: {status}")
    for w in warnings:
        print(f"  - {w}")

TB2AFC: PASS
Encoding: PASS
FIN2AFC: PASS
FINtimeline: PASS


### 2.2 Quick Look at Trial Counts per Subject x Session

In [11]:
trial_counts = (
    tb2afc.groupby(["subject", "session"])
    .size()
    .unstack(fill_value=0)
)
print("TB2AFC trial counts (subject x session):")
trial_counts

TB2AFC trial counts (subject x session):


session,04,05,06,07,08,09,10,11,12,13,14,15,16,17,18
subject,,,,,,,,,,,,,,,
03,42,76,78,76,78,76,78,76,78,76,78,76,78,76,36
04,42,76,78,76,78,76,78,76,78,76,78,76,78,76,36
05,42,76,78,76,78,76,78,76,78,76,78,76,78,76,36


---
## 3. Recognition Accuracy

Accuracy in the 2AFC task, broken down by experimental conditions.

### 3.1 Overall Accuracy by Subject

In [12]:
acc_subject = accuracy.accuracy_by_condition(tb2afc, group_cols=["subject"])
acc_subject

,subject,n_trials,n_correct,accuracy,se
0,03,1078,988,0.917,0.008
1,04,1078,880,0.816,0.012
2,05,1078,978,0.907,0.009


### 3.2 Accuracy by Encoding Condition (enCon)

In [13]:
acc_encon = accuracy.accuracy_by_condition(tb2afc, group_cols=["subject", "enCon"])

# Add readable labels
acc_encon["enCon_label"] = acc_encon["enCon"].map(ENCON_LABELS)
acc_encon

,subject,enCon,n_trials,n_correct,accuracy,se,enCon_label
0,03,1,336,300,0.893,0.017,single
1,03,2,364,342,0.940,0.013,repeats
2,03,3,378,346,0.915,0.014,triplets
3,04,1,336,253,0.753,0.024,single
4,04,2,364,307,0.843,0.019,repeats
5,04,3,378,320,0.847,0.019,triplets
6,05,1,336,291,0.866,0.019,single
7,05,2,364,336,0.923,0.014,repeats
8,05,3,378,351,0.929,0.013,triplets


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=acc_encon, x="enCon_label", y="accuracy", hue="subject",
            ax=ax, errorbar="se", alpha=0.8)
ax.axhline(0.5, ls="--", color="gray", alpha=0.5, label="chance")
ax.set_ylim(0, 1)
ax.set_title("Recognition Accuracy by Encoding Condition")
ax.set_xlabel("Encoding Condition")
ax.set_ylabel("Accuracy")
ax.legend(title="Subject")
plt.tight_layout()
plt.show()

### 3.3 Accuracy by Encoding x Retrieval Condition

In [ ]:
acc_full = accuracy.accuracy_by_condition(
    tb2afc, group_cols=["subject", "enCon", "reCon"]
)
acc_full["enCon_label"] = acc_full["enCon"].map(ENCON_LABELS)
acc_full["reCon_label"] = acc_full["reCon"].map(RECON_LABELS)
acc_full

In [ ]:
fig, axes = plt.subplots(1, len(SUBJECT_IDS), figsize=(5 * len(SUBJECT_IDS), 5),
                          sharey=True)
for ax, sub in zip(axes, SUBJECT_IDS):
    sub_data = acc_full[acc_full["subject"] == sub]
    sns.barplot(data=sub_data, x="enCon_label", y="accuracy",
                hue="reCon_label", ax=ax, alpha=0.8)
    ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
    ax.set_ylim(0, 1)
    ax.set_title(f"sub-{sub}")
    ax.set_xlabel("Encoding Condition")
    if sub == SUBJECT_IDS[0]:
        ax.set_ylabel("Accuracy")
    else:
        ax.set_ylabel("")
    ax.legend(title="Retrieval", fontsize=8)

fig.suptitle("Accuracy by enCon x reCon", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Signal Detection: d' (d-prime)

Sensitivity measure from signal detection theory. For 2AFC:

$$d' = \sqrt{2} \cdot \Phi^{-1}(p_{\text{correct}})$$

where $\Phi^{-1}$ is the inverse normal CDF.

In [ ]:
dprime_subject = accuracy.compute_sdt_2afc(tb2afc, group_cols=["subject"])
dprime_subject

In [ ]:
dprime_encon = accuracy.compute_sdt_2afc(
    tb2afc, group_cols=["subject", "enCon"]
)
dprime_encon["enCon_label"] = dprime_encon["enCon"].map(ENCON_LABELS)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=dprime_encon, x="enCon_label", y="dprime_2afc",
            hue="subject", ax=ax, alpha=0.8)
ax.axhline(0, ls="--", color="gray", alpha=0.5)
ax.set_title("d' by Encoding Condition")
ax.set_xlabel("Encoding Condition")
ax.set_ylabel("d' (2AFC)")
ax.legend(title="Subject")
plt.tight_layout()
plt.show()

### 4.1 Confidence-Accuracy Calibration

Does confidence predict accuracy? In a well-calibrated participant, "sure" responses should be more accurate than "maybe" responses.

In [ ]:
conf_acc = accuracy.confidence_accuracy_curve(
    tb2afc, group_cols=["subject"]
)
conf_acc["confidence_label"] = conf_acc["confidence_level"].map({1: "maybe", 2: "sure"})

fig, ax = plt.subplots(figsize=(6, 5))
sns.pointplot(data=conf_acc, x="confidence_label", y="accuracy",
              hue="subject", ax=ax, dodge=0.1)
ax.axhline(0.5, ls="--", color="gray", alpha=0.5, label="chance")
ax.set_ylim(0, 1)
ax.set_title("Confidence-Accuracy Calibration")
ax.set_xlabel("Confidence")
ax.set_ylabel("Accuracy")
ax.legend(title="Subject")
plt.tight_layout()
plt.show()

---
## 5. Reaction Times

RT distributions after filtering out fast guesses (<200 ms) and outliers (>3 SD from the mean).

In [ ]:
# Filter RT outliers
tb2afc_rt = preprocessing.filter_rt(tb2afc)
print(f"Trials after RT filtering: {len(tb2afc_rt):,} / {len(tb2afc):,}")

### 5.1 RT Summary Statistics

In [ ]:
rt_stats = rt.rt_summary(tb2afc_rt, group_cols=["subject"])
rt_stats

### 5.2 RT Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram by subject
for sub in SUBJECT_IDS:
    sub_data = tb2afc_rt[tb2afc_rt["subject"] == sub]["resp_RT"].dropna()
    axes[0].hist(sub_data, bins=40, alpha=0.5, label=f"sub-{sub}", density=True)
axes[0].set_title("RT Distribution by Subject")
axes[0].set_xlabel("RT (s)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Violin plot
sns.violinplot(data=tb2afc_rt, x="subject", y="resp_RT", ax=axes[1],
               inner="quart", alpha=0.7)
axes[1].set_title("RT Violin Plot")
axes[1].set_xlabel("Subject")
axes[1].set_ylabel("RT (s)")

plt.tight_layout()
plt.show()

### 5.3 RT by Accuracy (Correct vs. Incorrect)

In [ ]:
rt_acc = rt.rt_by_accuracy(tb2afc_rt, group_cols=["subject"])
rt_acc

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
rt_acc["accuracy_label"] = rt_acc["accurate"].map({True: "Correct", False: "Incorrect"})
sns.barplot(data=rt_acc, x="subject", y="mean_rt", hue="accuracy_label",
            ax=ax, alpha=0.8)
ax.set_title("Mean RT: Correct vs. Incorrect Trials")
ax.set_xlabel("Subject")
ax.set_ylabel("Mean RT (s)")
ax.legend(title="")
plt.tight_layout()
plt.show()

---
## 6. Learning Curves

How does recognition performance change across the 15 trial-based sessions?

### 6.1 Accuracy Learning Curve

In [ ]:
learn_acc = learning.session_learning_curve(
    tb2afc, metric_col="trial_accuracy", group_cols=["subject"]
)

fig, ax = plt.subplots(figsize=(10, 5))
for sub in SUBJECT_IDS:
    sub_data = learn_acc[learn_acc["subject"] == sub]
    ax.plot(sub_data["session_order"], sub_data["mean"], "o-", alpha=0.5,
            label=f"sub-{sub}")

# Group mean with error ribbon
group_mean = learn_acc.groupby("session_order")["mean"].agg(["mean", "sem"]).reset_index()
ax.plot(group_mean["session_order"], group_mean["mean"], "k-", linewidth=2.5,
        label="group mean")
ax.fill_between(group_mean["session_order"],
                group_mean["mean"] - group_mean["sem"],
                group_mean["mean"] + group_mean["sem"],
                alpha=0.2, color="black")

ax.axhline(0.5, ls="--", color="gray", alpha=0.4)
ax.set_title("Accuracy Learning Curve Across Sessions")
ax.set_xlabel("Session (0 = ses-04, 14 = ses-18)")
ax.set_ylabel("Mean Accuracy")
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 6.2 d' Learning Curve

In [ ]:
learn_dp = learning.session_dprime_curve(tb2afc, group_cols=["subject"])

fig, ax = plt.subplots(figsize=(10, 5))
for sub in SUBJECT_IDS:
    sub_data = learn_dp[learn_dp["subject"] == sub]
    ax.plot(sub_data["session_order"], sub_data["dprime_2afc"], "o-", alpha=0.5,
            label=f"sub-{sub}")

group_mean = learn_dp.groupby("session_order")["dprime_2afc"].agg(["mean", "sem"]).reset_index()
ax.plot(group_mean["session_order"], group_mean["mean"], "k-", linewidth=2.5,
        label="group mean")
ax.fill_between(group_mean["session_order"],
                group_mean["mean"] - group_mean["sem"],
                group_mean["mean"] + group_mean["sem"],
                alpha=0.2, color="black")

ax.axhline(0, ls="--", color="gray", alpha=0.4)
ax.set_title("d' Learning Curve Across Sessions")
ax.set_xlabel("Session (0 = ses-04, 14 = ses-18)")
ax.set_ylabel("d' (2AFC)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 6.3 Learning by Encoding Condition

Do different encoding conditions (single vs. repeats vs. triplets) show different learning trajectories?

In [ ]:
learn_cond = learning.compare_conditions_over_sessions(
    tb2afc, condition_col="enCon", metric_col="trial_accuracy"
)
learn_cond["condition_label"] = learn_cond["condition"].map(ENCON_LABELS)

# Group mean across subjects
cond_group = (
    learn_cond.groupby(["session_order", "condition_label"])["mean"]
    .agg(["mean", "sem"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
for cond, grp in cond_group.groupby("condition_label"):
    ax.plot(grp["session_order"], grp["mean"], "o-", label=cond, linewidth=2)
    ax.fill_between(grp["session_order"],
                    grp["mean"] - grp["sem"],
                    grp["mean"] + grp["sem"],
                    alpha=0.15)

ax.axhline(0.5, ls="--", color="gray", alpha=0.4)
ax.set_title("Learning Curves by Encoding Condition")
ax.set_xlabel("Session (0 = ses-04, 14 = ses-18)")
ax.set_ylabel("Mean Accuracy")
ax.set_ylim(0.3, 1.0)
ax.legend(title="enCon")
plt.tight_layout()
plt.show()

---
## 7. Subsequent Memory Effect (SME)

Does encoding quality (1=weak, 2=moderate, 3=strong association) predict later recognition accuracy? This links **encoding ratings** (TBencoding) with **recognition outcomes** (TB2AFC) via `pairId`.

### 7.1 Encoding Rating Distribution

In [ ]:
enc_dist = encoding.encoding_rating_distribution(
    enc_df, group_cols=["subject"]
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=enc_dist, x="rating", y="proportion", hue="subject",
            ax=ax, alpha=0.8)
ax.set_title("Distribution of Encoding Quality Ratings")
ax.set_xlabel("Encoding Rating (1=weak, 2=moderate, 3=strong)")
ax.set_ylabel("Proportion")
ax.legend(title="Subject")
plt.tight_layout()
plt.show()

### 7.2 Subsequent Memory Effect: Recognition Accuracy by Encoding Rating

In [ ]:
sme = encoding.subsequent_memory_effect(
    enc_df, tb2afc, group_cols=["subject"]
)
sme

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=sme, x="encoding_rating", y="accuracy", hue="subject",
            ax=ax, errorbar="se", alpha=0.8)
ax.axhline(0.5, ls="--", color="gray", alpha=0.5, label="chance")
ax.set_ylim(0, 1)
ax.set_title("Subsequent Memory Effect")
ax.set_xlabel("Encoding Rating (1=weak, 2=moderate, 3=strong)")
ax.set_ylabel("Recognition Accuracy")
ax.legend(title="Subject")
plt.tight_layout()
plt.show()

---
## 8. Final Session Analysis (ses-30)

The final session includes a comprehensive recognition test (FIN2AFC) covering all previously studied items, plus a temporal judgment task (FINtimeline).

### 8.1 Session vs. Final Recognition Accuracy

How does recognition accuracy at the final comprehensive test compare to session-by-session performance?

In [ ]:
fin_comp = final_session.fin_vs_tb_accuracy(fin2afc, tb2afc)

fig, ax = plt.subplots(figsize=(12, 5))
for sub in SUBJECT_IDS:
    sub_data = fin_comp[fin_comp["subject"] == sub].sort_values("session")
    ax.plot(range(len(sub_data)), sub_data["accuracy"].values, "o-",
            alpha=0.6, label=f"sub-{sub}", markersize=5)

# Label x-axis with session sources
all_sources = sorted(fin_comp["source"].unique())
ax.set_xticks(range(len(all_sources)))
ax.set_xticklabels(all_sources, rotation=45, ha="right", fontsize=8)
ax.axhline(0.5, ls="--", color="gray", alpha=0.4)
ax.set_title("Recognition Accuracy: Session-by-Session vs. Final Test")
ax.set_ylabel("Accuracy")
ax.legend(title="Subject", fontsize=9)
plt.tight_layout()
plt.show()

### 8.2 Long-Term Retention: Item-Level Tracking

Track individual items from their first recognition test through the final session. `retention_delta` = final accuracy - initial accuracy.

In [ ]:
retention = final_session.long_term_retention_curve(tb2afc, fin2afc)
print(f"Matched items: {len(retention):,}")

retention_summary = (
    retention.groupby("subject")
    .agg(
        n_items=("pairId", "count"),
        mean_initial=("initial_accuracy", "mean"),
        mean_final=("final_accuracy", "mean"),
        mean_delta=("retention_delta", "mean"),
    )
    .reset_index()
)
retention_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Initial vs. final accuracy
ret_melted = retention_summary.melt(
    id_vars="subject", value_vars=["mean_initial", "mean_final"],
    var_name="timepoint", value_name="accuracy"
)
ret_melted["timepoint"] = ret_melted["timepoint"].map({
    "mean_initial": "Initial", "mean_final": "Final"
})
sns.barplot(data=ret_melted, x="subject", y="accuracy", hue="timepoint",
            ax=axes[0], alpha=0.8)
axes[0].axhline(0.5, ls="--", color="gray", alpha=0.5)
axes[0].set_ylim(0, 1)
axes[0].set_title("Initial vs. Final Recognition Accuracy")
axes[0].set_ylabel("Accuracy")
axes[0].legend(title="")

# Retention delta distribution
for sub in SUBJECT_IDS:
    sub_data = retention[retention["subject"] == sub]
    axes[1].hist(sub_data["retention_delta"], bins=5, alpha=0.5,
                 label=f"sub-{sub}")
axes[1].axvline(0, ls="--", color="gray", alpha=0.5)
axes[1].set_title("Distribution of Retention Delta")
axes[1].set_xlabel("Final - Initial Accuracy")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

### 8.3 Timeline Temporal Judgments

In the FINtimeline task, participants judge *when* in the study each stimulus was encountered (0=early, 1=late).

In [ ]:
tl_cond = final_session.timeline_by_condition(timeline)
tl_cond

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of timeline responses
for sub in SUBJECT_IDS:
    sub_data = timeline[timeline["subject"] == sub]
    resp = pd.to_numeric(sub_data["timeline_resp"], errors="coerce").dropna()
    axes[0].hist(resp, bins=20, alpha=0.5, label=f"sub-{sub}", density=True)
axes[0].set_title("Timeline Response Distribution")
axes[0].set_xlabel("Timeline Response (0=early, 1=late)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Mean timeline response by enCon
tl_cond["enCon_label"] = tl_cond["enCon"].map(ENCON_LABELS)
tl_cond["reCon_label"] = tl_cond["reCon"].map(RECON_LABELS)
sns.barplot(data=tl_cond, x="enCon_label", y="mean_resp", hue="reCon_label",
            ax=axes[1], alpha=0.8)
axes[1].set_title("Mean Timeline Response by Condition")
axes[1].set_xlabel("Encoding Condition")
axes[1].set_ylabel("Mean Timeline Response")
axes[1].legend(title="Retrieval")

plt.tight_layout()
plt.show()

---
## 9. Summary Dashboard

A compact overview of key findings across all analyses.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Overall accuracy by subject
sns.barplot(data=acc_subject, x="subject", y="accuracy", ax=axes[0, 0], alpha=0.8)
axes[0, 0].axhline(0.5, ls="--", color="gray", alpha=0.4)
axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_title("Overall Accuracy")

# (0,1) d' by subject
sns.barplot(data=dprime_subject, x="subject", y="dprime_2afc", ax=axes[0, 1], alpha=0.8)
axes[0, 1].axhline(0, ls="--", color="gray", alpha=0.4)
axes[0, 1].set_title("Overall d'")
axes[0, 1].set_ylabel("d' (2AFC)")

# (0,2) RT by subject
sns.barplot(data=rt_stats, x="subject", y="mean_rt", ax=axes[0, 2], alpha=0.8)
axes[0, 2].set_title("Mean RT")
axes[0, 2].set_ylabel("RT (s)")

# (1,0) Accuracy learning curve
for sub in SUBJECT_IDS:
    sub_data = learn_acc[learn_acc["subject"] == sub]
    axes[1, 0].plot(sub_data["session_order"], sub_data["mean"], "o-", alpha=0.5,
                    markersize=4)
axes[1, 0].axhline(0.5, ls="--", color="gray", alpha=0.4)
axes[1, 0].set_title("Learning Curve")
axes[1, 0].set_xlabel("Session")
axes[1, 0].set_ylabel("Accuracy")

# (1,1) SME
sns.barplot(data=sme, x="encoding_rating", y="accuracy", ax=axes[1, 1],
            errorbar="se", alpha=0.8)
axes[1, 1].axhline(0.5, ls="--", color="gray", alpha=0.4)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_title("Subsequent Memory Effect")
axes[1, 1].set_xlabel("Encoding Rating")
axes[1, 1].set_ylabel("Recognition Accuracy")

# (1,2) Confidence calibration
sns.pointplot(data=conf_acc, x="confidence_level", y="accuracy",
              hue="subject", ax=axes[1, 2], dodge=0.1)
axes[1, 2].axhline(0.5, ls="--", color="gray", alpha=0.4)
axes[1, 2].set_ylim(0, 1)
axes[1, 2].set_title("Confidence Calibration")
axes[1, 2].set_xlabel("Confidence (1=maybe, 2=sure)")
axes[1, 2].set_ylabel("Accuracy")
axes[1, 2].legend(title="Subject", fontsize=7)

fig.suptitle("MMMData Behavioral Analysis Summary", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Export Results

Save all summary tables and figures to the derivatives directory. Uncomment the cells below to write to disk.

In [ ]:
OUTPUT_DIR = BIDS_ROOT / "derivatives" / "behavioral_analysis"
print(f"Output directory: {OUTPUT_DIR}")
print(f"Exists: {OUTPUT_DIR.exists()}")

# Uncomment to create output directory and save results:
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# (OUTPUT_DIR / "group").mkdir(exist_ok=True)
# (OUTPUT_DIR / "figures").mkdir(exist_ok=True)

In [ ]:
# Uncomment to save summary tables as TSV:

# tables = {
#     "accuracy_by_subject.tsv": acc_subject,
#     "accuracy_by_enCon.tsv": acc_encon.drop(columns=["enCon_label"]),
#     "accuracy_by_enCon_reCon.tsv": acc_full.drop(columns=["enCon_label", "reCon_label"]),
#     "dprime_by_subject.tsv": dprime_subject,
#     "dprime_by_enCon.tsv": dprime_encon.drop(columns=["enCon_label"]),
#     "rt_summary.tsv": rt_stats,
#     "rt_by_accuracy.tsv": rt_acc.drop(columns=["accuracy_label"], errors="ignore"),
#     "learning_curve_accuracy.tsv": learn_acc,
#     "learning_curve_dprime.tsv": learn_dp,
#     "sme_by_encoding_rating.tsv": sme,
#     "fin_comparison.tsv": fin_comp,
#     "timeline_by_condition.tsv": tl_cond.drop(columns=["enCon_label", "reCon_label"], errors="ignore"),
#     "retention_summary.tsv": retention_summary,
# }
#
# for fname, df in tables.items():
#     path = OUTPUT_DIR / "group" / fname
#     df.to_csv(path, sep="\t", index=False, na_rep="n/a")
#     print(f"Saved: {path}")